# 10: Regularized linear estimation and calibration

![Linear estimation pipeline](../images/10_regularized_linear_estimation.svg)

**Learning goals:** build a leakage-safe design matrix, derive and solve ridge regression, inspect conditioning and intercept treatment, fit weighted logistic models, implement stable softmax, and fit a temperature by held-out negative log likelihood.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 10
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print(f'NumPy {np.__version__}, synthetic data only')

## 1. Standardization and one-hot encoding

We fit both transforms on training data only. `StandardScaler` stores training means and scales. `OneHotEncoder(handle_unknown='ignore')` gives an unseen category an all-zero indicator block. Dropping one level avoids exact dependence with an intercept, but it also makes an unknown category indistinguishable from the dropped reference category.

In [ ]:
numeric_train = np.array([[1.0, 10.0], [2.0, 12.0], [3.0, 14.0], [4.0, 16.0]])
category_train = np.array([['red'], ['blue'], ['red'], ['green']])
numeric_test = np.array([[5.0, 18.0]])
category_test = np.array([['purple']])  # unseen during fitting
scaler = StandardScaler().fit(numeric_train)
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False).fit(category_train)
design_train = np.column_stack([scaler.transform(numeric_train), encoder.transform(category_train)])
design_test = np.column_stack([scaler.transform(numeric_test), encoder.transform(category_test)])
print('design shapes:', design_train.shape, design_test.shape)
print('unseen category block:', design_test[0, 2:])
assert np.allclose(design_train[:, :2].mean(axis=0), 0.0)
assert np.all(design_test[0, 2:] == 0.0)

## 2. Ridge regression and conditioning

Nearly copied columns make `X.T @ X` ill-conditioned. Ridge shifts its eigenvalues by `alpha`. We augment `X` with ones and use a diagonal penalty mask whose first entry is zero, so the intercept is not shrunk. `np.linalg.solve` avoids an explicit inverse.

In [ ]:
n = 200
x1 = rng.normal(size=n)
x2 = x1 + 1e-3 * rng.normal(size=n)
x = np.column_stack([x1, x2])
y = 3.0 + 2.0 * x1 + rng.normal(scale=0.2, size=n)
xa = np.column_stack([np.ones(n), x])
penalty = np.diag([0.0, 1.0, 1.0])
alpha = 1.0
unregularized_condition = np.linalg.cond(x.T @ x)
regularized_condition = np.linalg.cond(x.T @ x + alpha * np.eye(2))
theta = np.linalg.solve(xa.T @ xa + alpha * penalty, xa.T @ y)
print(f'condition before={unregularized_condition:.2e}, after={regularized_condition:.2e}')
print('intercept and coefficients:', np.round(theta, 3))
assert regularized_condition < unregularized_condition
assert abs(theta[0] - 3.0) < 0.1

The individual coefficients are unstable without regularization because many pairs produce almost the same prediction. Ridge prefers a smaller-norm pair. The fitted sum of the two coefficients matters more than either coefficient alone.

## 3. Multiclass logistic regression and class weights

The synthetic classes are imbalanced. `class_weight='balanced'` weights each class inversely to its training frequency. This changes the fitting objective, so it should be judged with a class-sensitive metric such as balanced accuracy.

In [ ]:
X, labels = make_classification(
    n_samples=1800, n_features=8, n_informative=6, n_redundant=1, n_classes=3,
    n_clusters_per_class=1, weights=[0.65, 0.25, 0.10], class_sep=1.2, random_state=SEED,
)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, labels, test_size=0.4, stratify=labels, random_state=SEED
)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED
)
scale = StandardScaler().fit(X_train)
X_train_s, X_cal_s, X_test_s = map(scale.transform, [X_train, X_cal, X_test])
plain = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED).fit(X_train_s, y_train)
weighted = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_train_s, y_train)
plain_bal = balanced_accuracy_score(y_test, plain.predict(X_test_s))
weighted_bal = balanced_accuracy_score(y_test, weighted.predict(X_test_s))
print(f'balanced accuracy: plain={plain_bal:.3f}, weighted={weighted_bal:.3f}')
assert plain.coef_.shape == weighted.coef_.shape == (3, 8)

## 4. Stable softmax and temperature scaling

Subtracting each row maximum prevents overflow and changes no probability. To make temperature scaling visible, we multiply fitted logits by 2.5, simulating an overconfident model. We select a positive temperature only by held-out negative log likelihood. NLL is a proper scoring rule, but improving its empirical value on the fitting set does not by itself establish calibrated probabilities. This notebook does not include a reliability diagnostic.

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)

cal_logits = plain.decision_function(X_cal_s) * 2.5
test_logits = plain.decision_function(X_test_s) * 2.5
temperatures = np.unique(np.append(np.geomspace(0.25, 8.0, 160), 1.0))
cal_losses = np.array([log_loss(y_cal, softmax(cal_logits / t), labels=plain.classes_) for t in temperatures])
temperature = temperatures[np.argmin(cal_losses)]
cal_nll_before = log_loss(y_cal, softmax(cal_logits), labels=plain.classes_)
before = softmax(test_logits)
after = softmax(test_logits / temperature)
nll_before = log_loss(y_test, before, labels=plain.classes_)
nll_after = log_loss(y_test, after, labels=plain.classes_)
print(f'best T={temperature:.2f}; calibration NLL before={cal_nll_before:.3f}, fitted={cal_losses.min():.3f}')
print(f'observed finite test NLL before={nll_before:.3f}, after={nll_after:.3f}')
assert np.allclose(after.sum(axis=1), 1.0)
assert np.array_equal(before.argmax(axis=1), after.argmax(axis=1))
assert cal_losses.min() <= cal_nll_before + 1e-12  # guaranteed on the fitted partition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), constrained_layout=True)
axes[0].semilogx(temperatures, cal_losses, color='#367cad')
axes[0].axvline(temperature, color='#c75b4b', linestyle='--')
axes[0].set(xlabel='temperature', ylabel='held-out NLL', title='Choose T by a proper scoring rule')
axes[1].hist(before.max(axis=1), bins=12, alpha=0.65, label='before', color='#c75b4b')
axes[1].hist(after.max(axis=1), bins=12, alpha=0.65, label='after', color='#27896f')
axes[1].set(xlabel='maximum probability', title='Confidence distribution shifts')
axes[1].legend()
plt.show()

## Exercises and takeaways

1. Set `penalty[0, 0] = 1`. **Check:** large regularization now pulls the intercept toward zero.
2. Compare `np.linalg.inv(A) @ b` with `np.linalg.solve(A, b)`. **Check:** solutions are close here, but `solve` avoids unnecessary inversion and is the preferred API.
3. Verify softmax invariance by adding 1,000 to every logit. **Check:** the stable implementation returns unchanged probabilities.
4. Fit temperature on test labels. **Check:** the number may look better, but the test set is no longer an unbiased final evaluation.

**Takeaways:** preprocessing must be fitted on training data, ridge shifts unstable eigenvalues and normally excludes the intercept, class weighting changes the objective, and stable softmax handles extreme logits. Temperature fitting requires separate held-out data. A claim of empirical calibration additionally requires reliability evidence with uncertainty on untouched data.

## Continue learning

[Previous notebook: 09](09_eigenspectra_and_effective_rank.ipynb) | [Lecture](../lectures/10_regularized_linear_estimation.md) | [Curriculum](../README.md) | [Next notebook: 11](11_factorial_state_spaces.ipynb)